# Lab 3: Comparing Models Fairly, and Checking We Can Trust the Winner

**Time:** about 75 minutes

In Lab 1 you trained several models and compared them in a table of numbers. That table had a problem: each number
answered a slightly different question, and none of them answered the question the review team actually asks:

> *"We can check about **5% of claims** before payment. If we check the ones your model says are riskiest,
> how many of the problem claims will we catch?"*

In this lab you will:

1. write **one function that tests any model in exactly the same way**, including that "5% question", so the
   comparison between models is fair;
2. use it to compare four models and pick one;
3. check that the chosen model works in more than one year, and that it relies on sensible things;
4. check whether it treats groups of members fairly;
5. fall into, and then get out of, a trap that catches experienced teams: a model that looks brilliant and is useless;
6. save the chosen model together with a short description of it.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **irregular claim** | a claim the audit later found to be a problem (overpaid, not allowed, false documents, fraud) |
| **risk score** | a number from 0 to 1 the model gives each claim; higher means "more likely to be irregular" |
| **capacity** | how many claims the review team can actually check; here, 5% of all claims |
| **precision** | of the claims the team checks, the share that really are irregular. High precision = little wasted effort |
| **recall** | of all the irregular claims, the share the team catches. High recall = little money lost |
| **threshold** | the score above which a claim gets checked |
| **ROC-AUC** | one number for how well the scores put irregular claims above regular ones. 0.5 = no better than guessing, 1.0 = perfect |
| **PR-AUC** | a similar number that pays most attention to the top of the list; better when problem cases are rare |
| **backtest** | testing a model on several past years, as if it had been built at the start of each one |
| **data leakage** | when a model is accidentally given information it won't have in real use, so its test results look far too good |

## Where this lab fits

**What you've done so far.** On Day 1 you cleaned the Fund's data. In Lab 1 you trained four models to score claims
for risk and compared them in one table. In Lab 2 you found groups of members with no target at all.

**What you'll do now.** You'll go back to the claims models and ask the questions a careful reviewer, an auditor or a
manager would ask before any model is used: *Is the comparison fair? Does the model help the team as it actually
works, checking 5% of claims? Would the winner have won in other years too? Does it rely on sensible things? Does it
treat groups of members fairly? Could its results be too good to be true?*

**Why it matters.** It's easy to build a model that scores well in a notebook. It's much harder to build one you can
**defend**: to the review team who'll work its list, to the auditor who asks how it was tested, and to a member who
asks why their claim was held. This lab is about the checks that make that possible.

**What happens next.** The model you save at the end comes back twice: on Day 3 you'll see whether a neural network
beats it, and on Day 10 you'll deploy it, watch it for changes, and take its description to a governance review.

### Theory: what makes a comparison between models fair

Two models can only be compared if everything except the model is the same. That means:

| Must be the same | Why |
|---|---|
| **training data** | a model trained on more or better data has an unfair head start |
| **test data** | some test sets are easier than others |
| **preparation steps** | learned from the training data only, every time |
| **measures** | each measure answers a different question; compare like with like |
| **random seed** | so a lucky random start doesn't decide the winner |

Doing all of that by hand for every model is where mistakes creep in: one model gets tested on a slightly different
set, another gets its scaler fitted on all the data. Writing it **once, as a function**, and putting every model
through that same function makes the comparison fair by construction. That's what Section 3 does.

## 0. Imports and settings

This cell loads the libraries the lab uses. You saw most of them in Lab 1. The new ones are:

- `time`, to measure how long each model takes to train;
- `json` and `joblib`, to save the chosen model and its description to files;
- `permutation_importance`, a way of asking a model which inputs it depends on (Section 6).

The two lines starting with `os.environ` and `warnings` only hide harmless messages, so the output stays readable.

In [ ]:
import os
import json                        # reads and writes JSON text files
import time                        # measures how long things take
import warnings
from pathlib import Path           # builds file and folder paths

# Windows 11 no longer has a tool ("wmic") some libraries use to count CPU cores. They print a long harmless error
# about it; telling them how many cores to use skips that check.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)

# Hide "this will change in a future version" notices from the libraries.
warnings.filterwarnings("ignore")

import joblib                      # saves a trained model to a file, and loads it back
import numpy as np                 # fast maths on arrays of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts

# scikit-learn: preparing the data, the models, and the measurements.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

# Two boosted-tree libraries.
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

Three settings used throughout:

- show wider tables when printing;
- `SEED` is any fixed number. Models that use randomness give the same result every time you run them with the same
  seed, so your numbers match your neighbour's;
- `CAPACITY = 0.05` is the review team's capacity: they can check 5% of claims.

In [ ]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

SEED = 42
CAPACITY = 0.05      # the review team can check 5% of claims before payment

> **Where we are.** The libraries are loaded, and the two settings that run through the whole lab are fixed: the
> random seed and the review team's capacity of 5%. Next you'll rebuild Lab 1's data so this notebook stands on its
> own.

## 1. The same data and features as Lab 1

This notebook works on its own, so it repeats the data preparation from Lab 1. (In a real project this code would
live in one shared file that every notebook uses, so it's only written once. Day 1's pipeline lab was about exactly
that.)

First read the claims file. `parse_dates` turns the `claim_date` column into real dates, so you can compare them with
`<` and `>=` later.

In [ ]:
claims = pd.read_csv("claims_review_history.csv", parse_dates=["claim_date"], dtype={"claim_no": "string"})

print(claims.shape)          # (rows, columns)
claims.head(3)

Now add the same extra columns ("features") as Lab 1. Each one turns raw data into something a claims officer would
recognise as a warning sign. Each line only uses values from the same row, so nothing is learned from other claims.

In [ ]:
# How big is the claim compared with the member's own monthly wage? (2.0 = twice their wage)
claims["amount_to_wage_ratio"] = claims["amount_claimed_tzs"] / claims["avg_wage_last_6m_tzs"]

# Were the bank details ever changed? 1 = yes, 0 = no. (An empty value means "never changed".)
claims["bank_ever_changed"] = claims["bank_details_changed_days_ago"].notna().astype(int)

# Were they changed in the 60 days before the claim? This was the strongest warning sign in Lab 1.
claims["bank_changed_60d"] = (claims["bank_details_changed_days_ago"] <= 60).astype(int)

# Has the member claimed before?
claims["has_previous_claim"] = (claims["previous_claims"] > 0).astype(int)

# Empty means "never": replace with 9999 days, i.e. "a very long time ago".
claims["days_since_last_claim"] = claims["days_since_last_claim"].fillna(9999)
claims["bank_details_changed_days_ago"] = claims["bank_details_changed_days_ago"].fillna(9999)

# Claim amounts range from thousands to millions. log1p squeezes them onto a smaller scale (log of 1 + amount).
claims["log_amount"] = np.log1p(claims["amount_claimed_tzs"])

claims[["amount_to_wage_ratio", "bank_changed_60d", "has_previous_claim", "log_amount"]].head(3)

Write down which columns the model may use. Numbers go in one list and text categories in another, because they're
prepared differently (Section 1, next cell).

In [ ]:
# Number columns.
NUMERIC = [
    "member_age", "months_contributed", "months_contributed_last_12", "log_amount", "amount_to_wage_ratio",
    "employer_arrears_months_12m", "days_since_registration", "previous_claims", "days_since_last_claim",
    "documents_missing_at_lodging", "bank_details_changed_days_ago", "bank_ever_changed", "bank_changed_60d",
    "has_previous_claim",
]

# Text (category) columns.
CATEGORICAL = ["benefit_type", "channel", "employer_sector", "gender"]

FEATURES = NUMERIC + CATEGORICAL
print(len(FEATURES), "columns go into the model")

Split by date, exactly as in Lab 1: the model learns from claims up to the end of 2024 and is tested on 2025 claims,
which it has never seen. That copies real use, where the model always scores claims that come *after* the ones it
learned from.

- `X` (capital X) is the table of inputs;
- `y` is the answer we want the model to predict: `irregular` (1 = problem claim, 0 = fine).

In [ ]:
# Claims before 2025 for learning ("training"); claims from 2025 for testing.
train = claims[claims["claim_date"] < "2025-01-01"]
test = claims[claims["claim_date"] >= "2025-01-01"]

X_train = train[FEATURES]
y_train = train["irregular"]
X_test = test[FEATURES]
y_test = test["irregular"]

print(len(train), "training claims,", len(test), "test claims")
print("share irregular in the test set:", round(y_test.mean(), 3))

### Two small functions to avoid repeating ourselves

You'll build several models below, and each one needs the same **preparation steps** from Lab 1:

- fill empty values (with the middle value for numbers, the most common value for categories);
- optionally put all numbers on the same scale (logistic regression needs this; tree models don't);
- turn each category into 0/1 columns ("one-hot encoding").

Rather than copy those steps four times, the function `make_preprocessor` builds them. Its inputs say which columns
are numbers, which are categories, and whether to scale.

In [ ]:
def make_preprocessor(numeric_columns, category_columns, scale_numbers):
    """Build the preparation steps from Lab 1: fill gaps, optionally scale numbers, one-hot encode categories."""

    # Steps for the number columns.
    if scale_numbers:
        numeric_steps = Pipeline([
            ("impute", SimpleImputer(strategy="median")),   # fill gaps with the middle value
            ("scale", StandardScaler()),                    # put every number on the same scale
        ])
    else:
        numeric_steps = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
        ])

    # Steps for the category columns.
    category_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),   # fill gaps with the most common value
        ("onehot", OneHotEncoder(handle_unknown="ignore")),    # one 0/1 column per category value
    ])

    # Apply each set of steps to its own columns.
    return ColumnTransformer([
        ("num", numeric_steps, numeric_columns),
        ("cat", category_steps, category_columns),
    ])

XGBoost (eXtreme Gradient Boosting) has many settings, and you'll create it several times. `make_xgboost` keeps all
its settings in one place, so every XGBoost model in this lab is identical. The settings are the ones from Lab 1:

- `n_estimators=400`: build 400 small decision trees, each one correcting the mistakes of the ones before;
- `max_depth=4`: each tree asks at most 4 yes/no questions;
- `learning_rate=0.05`: each new tree only makes a small correction (small steps are safer);
- `subsample` / `colsample_bytree = 0.8`: each tree sees a random 80% of the claims and 80% of the columns;
- `scale_pos_weight`: how much more a mistake on an irregular claim should count, because they're rare.

In [ ]:
def make_xgboost(scale_pos_weight):
    """An XGBoost model with the Lab 1 settings."""
    return XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",      # the measure XGBoost uses while learning
        random_state=SEED,
        n_jobs=4,                   # use 4 CPU cores
    )

> **Where we are.** The data, features, time split and preparation steps are exactly as in Lab 1. Now you'll replace
> Lab 1's measures, which assumed a cut-off of 0.5 that nobody would really use, with measures that match how the
> review team works.

### Theory: measuring a model where it's used

In Lab 1 you measured precision and recall at a threshold of 0.5. But no one chose 0.5 for a business reason; it's
just the halfway point. The review team doesn't think in thresholds. They think in **workload**: *"we can check about
5% of claims"*.

So turn the question around. Sort all claims by score, take the top 5%, and measure *that* list. These are called
**precision at k** and **recall at k**, where *k* is the size of the list the team actually works through. The
threshold then falls out of the capacity, instead of the other way round.

This is a general habit worth keeping: **measure a model at the operating point where it'll really be used**, not at
a default.

## 2. What "5% capacity" means, worked out by hand

If the team checks the riskiest 5% of claims, two numbers matter:

- **Precision at 5%:** of the claims they check, what share really are irregular? This is how well their time is spent.
- **Recall at 5%:** of *all* the irregular claims, what share did they catch? This is how much money is protected.

Neither number depends on choosing a cut-off like 0.5. The cut-off is simply whatever score puts a claim in the top 5%.

To work it out, you first need a model with scores. Train the XGBoost model from Lab 1.

In [ ]:
# About 10 regular claims for every irregular one in the training data. Tell XGBoost to count each
# irregular claim about 10 times as much, so it doesn't just ignore them.
spw = (y_train == 0).sum() / (y_train == 1).sum()
print("regular claims per irregular claim:", round(spw, 1))

# Preparation + model, trained on the training claims.
xgb = Pipeline([
    ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=False)),
    ("model", make_xgboost(spw)),
])
xgb.fit(X_train, y_train)

# A risk score for every test claim. predict_proba gives two columns (chance of 0, chance of 1); [:, 1] keeps the second.
p_xgb = xgb.predict_proba(X_test)[:, 1]
print("first five scores:", p_xgb[:5].round(3))

Now the four steps, one at a time.

**Step 1.** Put each test claim's true answer next to its score, and sort so the riskiest claim is at the top. This
sorted list is exactly the order the review team would work through.

In [ ]:
ranked = pd.DataFrame({"irregular": y_test.values, "score": p_xgb})
ranked = ranked.sort_values("score", ascending=False)      # highest score first
ranked.head(10)

**Step 2.** How many claims can the team check? 5% of the test claims, rounded to a whole number.

**Step 3.** Those top claims, from the sorted list, are the ones they check.

In [ ]:
n_review = round(CAPACITY * len(ranked))
print("claims the team can check:", n_review)

reviewed = ranked.head(n_review)        # the top n_review rows of the sorted list

**Step 4.** Count the irregular claims among those checked, and turn the counts into precision and recall. The lowest
score in the checked group is the **threshold**: any claim scoring at least that much would be checked.

In [ ]:
caught = reviewed["irregular"].sum()            # irregular claims among those checked (1s add up, 0s don't)
total_irregular = ranked["irregular"].sum()     # irregular claims in the whole test set

print("irregular claims caught:", caught, "out of", total_irregular)
print("precision at 5%:", round(caught / n_review, 3), "  <- share of checked claims that were irregular")
print("recall at 5%:   ", round(caught / total_irregular, 3), "  <- share of all irregular claims that were caught")
print("threshold (lowest score that gets checked):", round(reviewed["score"].min(), 3))

Read the result in plain words: *"If the team checks the 5% of claims this model ranks highest, about 8 in 10 of the
claims they open are real problems, and they catch a bit over 4 in 10 of all the problem claims."*

You'll need these numbers for every model, so the next cell puts the same four steps into a function called
`at_capacity`. Its inputs are the true answers, the scores and the capacity; it returns the threshold, precision and
recall.

In [ ]:
def at_capacity(y_true, risk_scores, capacity=CAPACITY):
    """If the team checks the top `capacity` share of claims by score: the threshold, precision and recall."""
    # Step 1: true answers next to scores, riskiest first.
    ranked = pd.DataFrame({"irregular": np.asarray(y_true), "score": risk_scores})
    ranked = ranked.sort_values("score", ascending=False)

    # Steps 2 and 3: how many can be checked, and which ones. max(1, ...) makes sure at least one is checked.
    n_review = max(1, round(capacity * len(ranked)))
    reviewed = ranked.head(n_review)

    # Step 4: count, and turn the counts into shares. float() and round() just make the numbers print neatly.
    caught = reviewed["irregular"].sum()
    return {
        "threshold": round(float(reviewed["score"].min()), 3),
        "precision_at_cap": round(float(caught / n_review), 3),
        "recall_at_cap": round(float(caught / ranked["irregular"].sum()), 3),
    }


# Check: the function gives the same numbers as the steps above.
at_capacity(y_test, p_xgb)

For comparison: if the team picked claims **at random**, 5% of claims would catch about 5% of the irregular ones,
with a precision equal to the overall irregular rate (about 10%). Any useful model must do much better than that.

In [ ]:
print(f"A random choice of {CAPACITY:.0%} of claims would catch about {CAPACITY:.0%} of irregular claims,")
print(f"and about {y_test.mean():.0%} of the claims checked would be irregular.")

> **Where we are.** You've worked out precision and recall at 5% capacity by hand, step by step, and wrapped those
> steps in `at_capacity`. You've also seen what random checking would give. Now you'll put everything (training,
> scoring and every measure) into one function, so all four models are tested in exactly the same way.

## 3. One function that tests every model the same way

To compare models fairly, every model must be trained on the same data, tested on the same data, and measured with
the same numbers. The function `evaluate_model` does all of that in one go. You give it a name and a model; it:

1. trains the model on the training claims, and times how long that takes;
2. scores the test claims;
3. measures the model in every way that matters here: the Lab 1 measures (accuracy, precision, recall and F1 at the
   usual cut-off of 0.5; ROC-AUC; PR-AUC) and the capacity measures from Section 2;
4. returns one row of results, plus the scores (you'll need them for charts).

Training time is included because a model that takes an hour to retrain is a real cost for the team that runs it.

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    """Train one model, score the test claims, and measure it. Returns (one row of results, the scores)."""

    # 1. Train, and time it.
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    # 2. Scores for the test claims, and yes/no decisions using the usual cut-off of 0.5.
    risk_scores = model.predict_proba(X_test)[:, 1]
    flagged = (risk_scores >= 0.5).astype(int)

    # 3. The capacity numbers (Section 2).
    capacity = at_capacity(y_test, risk_scores)

    # 4. Everything in one row.
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, flagged),                       # share of yes/no decisions that were right
        "precision@0.5": precision_score(y_test, flagged, zero_division=0),  # of claims scoring 0.5+, share irregular
        "recall@0.5": recall_score(y_test, flagged),                       # of irregular claims, share scoring 0.5+
        "f1@0.5": f1_score(y_test, flagged),                               # one number balancing the two above
        "roc_auc": roc_auc_score(y_test, risk_scores),                     # how well scores rank (0.5 = guessing)
        "pr_auc": average_precision_score(y_test, risk_scores),            # the same, focused on the top of the list
        "threshold": capacity["threshold"],
        "precision_at_cap": capacity["precision_at_cap"],
        "recall_at_cap": capacity["recall_at_cap"],
        "fit_seconds": fit_seconds,
    }
    return row, risk_scores

Now define the four models to compare. Each one is the preparation steps plus a model, joined in a `Pipeline` so the
preparation is learned from the training data only.

**Model 1: logistic regression.** A simple, explainable model: one weight per input. It needs the numbers scaled.
`class_weight="balanced"` makes it take the rare irregular claims seriously.

In [ ]:
logreg = Pipeline([
    ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=True)),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

**Model 2: a decision tree.** A chain of yes/no questions, at most 5 deep (`max_depth=5`), with at least 50 claims in
every final group (`min_samples_leaf=50`) so it can't memorise individual claims.

In [ ]:
tree = Pipeline([
    ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=False)),
    ("model", DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, class_weight="balanced", random_state=SEED)),
])

**Model 3: XGBoost**, with the settings from `make_xgboost`.

In [ ]:
xgb = Pipeline([
    ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=False)),
    ("model", make_xgboost(spw)),
])

**Model 4: LightGBM**, another boosted-tree library that works in a similar way to XGBoost. `num_leaves=15` limits how
detailed each tree can be; `verbose=-1` stops it printing progress messages.

In [ ]:
lgbm = Pipeline([
    ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=False)),
    ("model", LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=15, class_weight="balanced",
                             random_state=SEED, verbose=-1)),
])

Put each model through `evaluate_model`. Every call gives back one row of results and the model's scores. Then stack
the four rows into one table.

In [ ]:
row_lr, p_lr = evaluate_model("Logistic regression", logreg, X_train, y_train, X_test, y_test)
row_tree, p_tree = evaluate_model("Decision tree", tree, X_train, y_train, X_test, y_test)
row_xgb, p_xgb = evaluate_model("XGBoost", xgb, X_train, y_train, X_test, y_test)
row_lgbm, p_lgbm = evaluate_model("LightGBM", lgbm, X_train, y_train, X_test, y_test)

# One row per model. set_index("model") puts the model names down the left-hand side.
report = pd.DataFrame([row_lr, row_tree, row_xgb, row_lgbm])
report = report.set_index("model")
report.round(3)

> **Where we are.** All four models have been through `evaluate_model`, and the table shows which one helps the team
> most at today's 5% capacity. But 5% is today's number; staffing changes.

How to read this table:

- Start with the columns that matter to the review team: **`precision_at_cap`** and **`recall_at_cap`**. They answer
  the question the team actually asks.
- Then look at `roc_auc` and `pr_auc`, which describe the whole ranking.
- Notice that the model with the best **accuracy** isn't necessarily the best at 5% capacity. Accuracy measures yes/no
  decisions at a cut-off of 0.5, which nobody is going to use, so it's the least useful column here.

## 4. What if the team's capacity changes?

Next year the review team might grow, or shrink. A **gain chart** shows the trade-off for every possible capacity at
once:

- along the bottom: the share of claims checked (going down the sorted list from the riskiest);
- up the side: the share of all irregular claims caught so far.

A perfect model shoots straight up (the first claims checked are all irregular). Random checking follows the
diagonal line. The higher a model's curve, the better.

The function below works out one curve. `cumsum()` is a running total: as you go down the sorted list, it counts how
many irregular claims you've found so far.

In [ ]:
def gain_curve(y_true, risk_scores):
    """For every possible capacity: (share of claims checked, share of irregular claims caught)."""
    ranked = pd.DataFrame({"irregular": np.asarray(y_true), "score": risk_scores})
    ranked = ranked.sort_values("score", ascending=False)

    # Running count of irregular claims found, divided by the total: 0 at the start, 1 at the end.
    caught = ranked["irregular"].cumsum() / ranked["irregular"].sum()

    # Share of claims checked so far: 1/n, 2/n, ... n/n.
    share_reviewed = np.arange(1, len(ranked) + 1) / len(ranked)
    return share_reviewed, caught.values

Draw one curve per model, the diagonal for random checking, and a vertical line at today's 5% capacity. The chart
stops at 40% of claims checked, because no team will check more than that.

In [ ]:
plt.figure(figsize=(6.5, 4.5))

share, caught = gain_curve(y_test, p_lr)
plt.plot(share, caught, label="Logistic regression", color="#2a78d6")

share, caught = gain_curve(y_test, p_tree)
plt.plot(share, caught, label="Decision tree", color="#b5b4ad")

share, caught = gain_curve(y_test, p_xgb)
plt.plot(share, caught, label="XGBoost", color="#eb6834")

share, caught = gain_curve(y_test, p_lgbm)
plt.plot(share, caught, label="LightGBM", color="#1baf7a")

plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="random")                 # the diagonal
plt.axvline(CAPACITY, color="grey", lw=0.8)                              # today's capacity
plt.text(CAPACITY + 0.01, 0.05, "5% capacity", color="grey")
plt.xlim(0, 0.4)
plt.xlabel("share of claims checked")
plt.ylabel("share of irregular claims caught")
plt.title("Gain chart, 2025 claims")
plt.legend(frameon=False)
plt.show()

The same information as a table, for XGBoost at four different capacities. This is the table to take to a manager.

In [ ]:
rows = []
for capacity in [0.02, 0.05, 0.10, 0.20]:
    result = at_capacity(y_test, p_xgb, capacity)     # threshold, precision and recall at this capacity
    result["capacity"] = capacity
    rows.append(result)

cap_table = pd.DataFrame(rows)
cap_table[["capacity", "threshold", "precision_at_cap", "recall_at_cap"]]

This table starts a useful conversation with management: *"With today's team we catch this much. If you gave us two
more reviewers, we'd catch that much more."* That's a far better argument for resources than a single ROC-AUC number.

**Your turn (1).** The team says: *"We'd rather catch at least 60% of irregular claims, even if it means more work."*
Using the gain curve or `at_capacity`, what share of claims would they need to check with XGBoost? And with logistic
regression?

In [ ]:
# Your code here

> **Where we are.** You've compared the models at every capacity, not just 5%, and you have a table to take to a
> manager. All of that rests on one test year, 2025.

### Theory: why one test result isn't enough

A test score is a measurement, and every measurement has some **noise**. 2025's claims are one particular set of
claims; a different year might suit a different model. If XGBoost beats logistic regression by 0.02 on one year, you
can't tell whether that's a real difference or luck.

A **backtest** repeats the whole experiment several times, each time pretending you're standing at a different point in
the past:

```
build at start of 2022:  learn 2019-2021  →  test 2022
build at start of 2023:  learn 2019-2022  →  test 2023
build at start of 2024:  learn 2019-2023  →  test 2024
build at start of 2025:  learn 2019-2024  →  test 2025
```

Each model only ever learns from claims **before** the year it's tested on, exactly as in real use. If one model wins
every year, the difference is real. If the winner changes from year to year, treat the two as equal, and choose the
simpler one. You'll use the same idea for forecasts on Day 4.

## 5. Does the result hold in other years?

One test year is a single sample: 2025 might just happen to suit one model. A **backtest** repeats the comparison as if
you'd built the model at the start of each year from 2022 to 2025:

- learn from every claim **before** 1 January of that year;
- test on the claims **during** that year.

If the same model wins in every year, you can trust the choice. The loop below does this for XGBoost and logistic
regression. It trains 8 models, so it takes a few seconds.

In [ ]:
backtest_rows = []

for year in [2022, 2023, 2024, 2025]:
    # Claims before this year (to learn from) and during this year (to test on).
    before = claims[claims["claim_date"] < f"{year}-01-01"]
    during = claims[(claims["claim_date"] >= f"{year}-01-01") & (claims["claim_date"] < f"{year + 1}-01-01")]

    # A fresh XGBoost, with the irregular-claim weight worked out from this year's training claims.
    weight = (before["irregular"] == 0).sum() / (before["irregular"] == 1).sum()
    xgb_year = Pipeline([
        ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=False)),
        ("model", make_xgboost(weight)),
    ])
    row_x, _ = evaluate_model("XGBoost", xgb_year, before[FEATURES], before["irregular"],
                              during[FEATURES], during["irregular"])

    # A fresh logistic regression.
    lr_year = Pipeline([
        ("prep", make_preprocessor(NUMERIC, CATEGORICAL, scale_numbers=True)),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    row_l, _ = evaluate_model("Logistic regression", lr_year, before[FEATURES], before["irregular"],
                              during[FEATURES], during["irregular"])

    # Keep the numbers that matter for this year.
    backtest_rows.append({
        "year": year,
        "train_claims": len(before),
        "test_claims": len(during),
        "XGBoost roc_auc": row_x["roc_auc"],
        "XGBoost recall_at_cap": row_x["recall_at_cap"],
        "LogReg roc_auc": row_l["roc_auc"],
        "LogReg recall_at_cap": row_l["recall_at_cap"],
    })
    print("done", year)

bt = pd.DataFrame(backtest_rows).set_index("year")
bt.round(3)

### Theory: two questions about what a model relies on

Understanding a model's reasoning is called **interpretability** or **explainability**. It answers two different
questions:

- **Global:** *"What does the model rely on, across all claims?"* This tells you whether the model makes business
  sense, and is what this section covers.
- **Local:** *"Why did this particular claim get a high score?"* This is what a reviewer or a member needs. You wrote
  one by hand in Lab 1, and Day 10's governance pack produces them automatically.

Some models are **interpretable by design**: a logistic regression's weights *are* its reasoning. Others, like XGBoost,
are too complex to read directly, so you study them from the outside, by changing their inputs and watching the
output. Permutation importance (below) is one such method. It works for any model, which is why it's so widely used.

If XGBoost is ahead in every year, the ranking is reliable. If the winner swaps from year to year, the difference
between the two models is probably luck, and the simpler model should win the argument, because it's easier to
explain and maintain.

## 6. Looking inside the models

A model that scores well but relies on the wrong things will fail as soon as circumstances change, and it will be
impossible to defend when someone asks why their claim was held. So look at **what each model depends on**.

### Logistic regression: the weights

Logistic regression gives every input one weight (a "coefficient"). Because the numbers were scaled, the weights can be
compared with each other:

- a **positive** weight pushes the risk score **up** when that input is high;
- a **negative** weight pushes it **down**;
- the bigger the weight (ignoring its sign), the more that input matters.

The weights are stored inside the model step of the pipeline; the matching input names come from the preparation step.

In [ ]:
weights = logreg.named_steps["model"].coef_[0]                     # one weight per prepared column
names = logreg.named_steps["prep"].get_feature_names_out()         # the name of each prepared column
coefs = pd.Series(weights, index=names)

# The 15 weights furthest from zero (abs() ignores the sign), sorted from most negative to most positive.
largest = coefs.abs().sort_values(ascending=False).head(15).index
top15 = coefs[largest].sort_values()
top15.round(2)

The same 15 weights as a chart: orange bars push risk up, blue bars push it down. The `num__` and `cat__` at the start
of each name just say whether it came from a number or a category column; `cat__channel_ONLINE` means "the channel is
ONLINE".

In [ ]:
# Choose a colour for each bar: orange for positive weights, blue for negative ones.
colours = []
for weight in top15:
    if weight > 0:
        colours.append("#eb6834")
    else:
        colours.append("#2a78d6")

plt.figure(figsize=(7, 5))
plt.barh(top15.index, top15.values, color=colours)
plt.axvline(0, color="black", lw=0.6)
plt.title("Logistic regression: largest weights")
plt.show()

### XGBoost: two ways to measure what matters

XGBoost doesn't have one weight per input, so there are two common ways to ask what it depends on:

1. **Built-in importance:** XGBoost keeps count of how often, and how usefully, each input was used in its 400 trees.
   Quick, but it describes the training data, not new claims.
2. **Permutation importance:** take the test data, **shuffle one column** so its values no longer match their claims,
   and see how much worse the model gets. If shuffling `bank_changed_60d` makes the model much worse, the model really
   relies on it. Slower, but measured on claims the model hasn't seen, so it's the more honest of the two.

First, the built-in importance:

In [ ]:
built_in = pd.Series(xgb.named_steps["model"].feature_importances_,
                     index=xgb.named_steps["prep"].get_feature_names_out())

# The 12 most important, as a bar chart.
built_in.sort_values().tail(12).plot(kind="barh", color="#2a78d6", figsize=(6, 4.5))
plt.title("XGBoost built-in importance")
plt.show()

Now permutation importance. Each column is shuffled 5 times (`n_repeats=5`) and the drop in PR-AUC is averaged.
`scoring="average_precision"` is scikit-learn's name for PR-AUC. This takes a few seconds.

In [ ]:
perm = permutation_importance(xgb, X_test, y_test, scoring="average_precision", n_repeats=5,
                              random_state=SEED, n_jobs=1)

# The average drop for each original column.
perm_importance = pd.Series(perm.importances_mean, index=X_test.columns)

perm_importance.sort_values().tail(12).plot(kind="barh", color="#eb6834", figsize=(6, 4.5))
plt.title("Permutation importance (drop in PR-AUC when shuffled)")
plt.show()

Now the most important question in this lab:

> **Does what the model relies on make business sense?**

Go through the top inputs with someone who knows the claims process. A recent bank-detail change, missing documents,
employer arrears, claims lodged by an employer's HR office, a claim that's large compared with the member's wage: an
experienced claims officer would recognise all of these as warning signs. That's reassuring.

If you'd found the model leaning heavily on `gender`, or on `branch` (which is why branch was left out), you'd have a
problem, however good the scores were.

**Your turn (2).** Look at the logistic regression weights again. Find one input whose direction (pushing risk up or
down) surprises you. Is there a sensible business explanation, or does it look like the model has picked up something
odd? Write your conclusion below.

*Your answer here.*

> **Where we are.** The model's top inputs (bank changes, missing documents, employer arrears, the HR channel, the
> amount compared with wage) are ones a claims officer would recognise. That's good. But a model can rely on sensible
> things overall and still affect one group of members more than another.

### Theory: fairness, proxies, and what to measure

**Why removing a column isn't enough.** A model can treat groups differently even if it never sees the group column.
Other inputs act as **proxies** (stand-ins): only women claim maternity benefit, so `benefit_type` carries information
about gender. Region can stand in for ethnicity or income. So fairness has to be checked on **outcomes**, not just on
the list of inputs.

**What to compare.** For each group, look at:

- the **flag rate**: what share of the group's claims gets checked? This is the *burden* on the group.
- the **real irregular rate**: what share of the group's claims actually are irregular? This is the *justification*.

If a group's flag rate is higher *and* its irregular rate is higher, the difference may be justified, but it should be
written down. If its flag rate is higher and its irregular rate isn't, members of that group are being checked more
for no reason. That's the pattern to fix.

There's no single mathematical definition of "fair" that suits every case, and some definitions contradict each other.
That's why fairness is a decision for a governance group, informed by these numbers, not something the code decides.

## 7. Is it fair?

The model doesn't use gender *as a reason*, but gender is still one of its inputs, and other inputs can stand in for it
(for example, only women claim maternity benefit). Before the model is used, check whether the **burden of being
checked** falls unevenly on one group.

For each gender, compare three numbers:

- **claims:** how many claims the group lodged;
- **flagged_rate:** the share of the group's claims that would be checked (top 5% by score);
- **irregular_rate:** the share of the group's claims that really were irregular.

In [ ]:
# The score a claim needs to be in the top 5% (from the capacity numbers).
threshold = at_capacity(y_test, p_xgb)["threshold"]

# A small table: each test claim's gender, whether it was irregular, and whether it would be checked.
fairness = test[["claim_no", "gender", "irregular"]].copy()
fairness["flagged"] = p_xgb >= threshold

# For each gender: count the claims, and average the True/False and 0/1 columns to get shares.
fairness.groupby("gender").agg(
    claims=("claim_no", "count"),
    flagged_rate=("flagged", "mean"),
    irregular_rate=("irregular", "mean"),
).round(3)

How to read it:

- If one group is flagged more often **and** really has more irregular claims, the difference may be justified. Write
  the reason down.
- If one group is flagged more often but **doesn't** have more irregular claims, the model is putting extra burden on
  that group for no good reason. That needs fixing, and it goes to the governance review on Day 10.
- Watch benefit type too: maternity claims are all lodged by women, so anything that affects maternity claims affects
  women only.

**Your turn (3).** Drop `gender` from the inputs, retrain XGBoost with `evaluate_model`, and compare. Does performance
change? Do the flag rates by gender change?

In [ ]:
# Your code here

> **Where we are.** You've compared the models fairly, checked them across capacities and years, looked inside them,
> and checked fairness by gender. XGBoost is the strongest candidate. Before you save it, one more check, and it's
> the one that catches experienced teams.

## 8. A trap: a model that looks brilliant

A colleague sends you an "improved" model. They've added one extra input, `investigation_opened`, and the results are
remarkable. Build it the same way and see for yourself:

In [ ]:
# The same model, with one extra number column.
leaky_numeric = NUMERIC + ["investigation_opened"]

leaky = Pipeline([
    ("prep", make_preprocessor(leaky_numeric, CATEGORICAL, scale_numbers=False)),
    ("model", make_xgboost(spw)),
])
row_leaky, _ = evaluate_model("XGBoost + investigation_opened", leaky,
                              train[leaky_numeric + CATEGORICAL], y_train,
                              test[leaky_numeric + CATEGORICAL], y_test)

# Put the honest XGBoost and the "improved" one side by side.
comparison = pd.DataFrame([row_xgb, row_leaky]).set_index("model")
comparison[["roc_auc", "pr_auc", "precision_at_cap", "recall_at_cap"]].round(3)

Before celebrating, ask one question about every input: **would we know this value at the moment we need the score?**

The model is meant to score a claim **when it's lodged**, before payment. `investigation_opened` is only filled in
**after** the audit has found a problem. When a new claim arrives, it's always empty. So the model has learned
*"claims that were investigated turned out to be irregular"*: true, and useless. In real use its performance would fall
back to whatever the other inputs can manage, and nobody would understand why.

This is **data leakage**: information from the future (or from the answer itself) slipping into the training data. It's
one of the most common reasons a model that looked excellent fails in real use. The warning signs are:

- results that seem too good to be true (they usually are);
- one input that dominates the importance chart;
- an input whose meaning is "something that happened *because of* the outcome".

`amount_recovered_tzs` has the same problem: it's only above zero because the claim was found to be irregular.

**Your turn (4).** Go through every column in `claims_review_history.csv` and decide whether its value is known at the
moment a claim is lodged. Are there any you'd want to double-check with the people who run the claims system?

*Your answer here.*

> **Where we are.** You've seen how a leaked input makes a model look brilliant and be useless, and how to spot it.
> The XGBoost model from Section 3, without any leaked input, is the one to put forward.

### Theory: saving a model, and why it needs a description

**Saving the model.** A trained model lives in memory and disappears when the notebook closes. `joblib.dump` writes
it to a file: all the learned values of the preparation steps (medians, scales, category lists) and of the model (the
400 trees). Another program can load it and score new claims without retraining. Because the preparation steps are
in the same pipeline, they're saved too, so new claims are always prepared exactly as the training claims were.

**The model card.** A file of trees means nothing to the next person. A **model card** is a short, plain-language
record kept next to the model: what it's for, what it was trained on and when, which inputs it expects, how well it
did and on which test, the threshold that matches capacity, and its known limits. It's what an auditor asks for, and
what you'll take to the governance review on Day 10.

## 9. Choose a model, and save it with a description

Pick the model you'd put forward, and save it **together with** what someone needs to use it responsibly: what it was
trained on, which inputs it expects, how well it did, and the score threshold that matches the team's capacity.

First the model itself. `joblib.dump` writes the whole trained pipeline (preparation steps and model) to one file;
`joblib.load` reads it back later, ready to score new claims.

In [ ]:
Path("models").mkdir(exist_ok=True)          # create the folder if it doesn't exist
joblib.dump(xgb, "models/claims_risk_model.joblib")
print("saved models/claims_risk_model.joblib")

Now the description, often called a **model card**: plain facts about the model, written as a Python dictionary
(name -> value). Every value comes from what you measured above, not from memory.

In [ ]:
card = {
    "model": "XGBoost",
    "purpose": "Rank newly lodged claims for pre-payment review. Decision support only; a reviewer decides.",
    "target": "irregular (post-payment audit found overpayment, ineligibility, false documents or fraud)",
    "training_period": str(train["claim_date"].min().date()) + " to " + str(train["claim_date"].max().date()),
    "test_period": str(test["claim_date"].min().date()) + " to " + str(test["claim_date"].max().date()),
    "features_numeric": NUMERIC,
    "features_categorical": CATEGORICAL,
    "excluded_on_purpose": ["branch (location proxy)", "investigation_opened (leakage)", "amount_recovered_tzs (leakage)"],
    "review_capacity": CAPACITY,
    "score_threshold_at_capacity": row_xgb["threshold"],
    "test_metrics": {
        "roc_auc": round(float(row_xgb["roc_auc"]), 4),
        "pr_auc": round(float(row_xgb["pr_auc"]), 4),
        "precision_at_cap": row_xgb["precision_at_cap"],
        "recall_at_cap": row_xgb["recall_at_cap"],
    },
}

Save the card as a JSON file (a plain-text format any system can read) next to the model, and print it.
`indent=2` lays it out over several lines so it's easy to read.

In [ ]:
card_text = json.dumps(card, indent=2)
Path("models/claims_risk_model_card.json").write_text(card_text)
print(card_text)

You'll come back to this model twice. On Day 3 you'll ask whether a neural network does any better. On Day 10 you'll
put it behind an API (application programming interface: a way for one program to call another), watch it for
changes over time, and take this card to a governance review.

## Looking back, looking ahead

**What you did in this lab.** You took the models from Lab 1 and put them through the checks a careful team runs
before any model is used:

1. one function that tests every model the same way, so the comparison is fair;
2. measures at the review team's real capacity, and a gain chart for other capacities;
3. a backtest over four years, so the choice doesn't rest on one lucky year;
4. a look inside the models, to check they rely on things that make business sense;
5. a fairness check by gender, on outcomes, not just inputs;
6. a model that looked brilliant because of a leaked input, and how to spot one;
7. the chosen model saved with a model card.

**What you did today.** Day 2 took you from clean data to a model you can defend: supervised learning for claims
risk (Lab 1), unsupervised learning for member segments (Lab 2), and the checks that turn a good score into a
trustworthy model (Lab 3).

**Tomorrow (Day 3).** Three new problems. Can a neural network beat this XGBoost model? How do you build features from
millions of contribution records without running out of memory? And six months from now, can you prove exactly which
data and code produced a model?

---

## Answers to the "Your turn" exercises

**(1)** How much would the team need to check to catch 60%? Go down the gain curve until the share caught first
reaches 0.60, and read off the share checked at that point. `np.argmax(caught >= 0.60)` finds the position of the first
point where that's true.

In [ ]:
# XGBoost.
share, caught = gain_curve(y_test, p_xgb)
first = np.argmax(caught >= 0.60)            # position of the first point where 60% or more is caught
print(f"XGBoost: check {share[first]:.1%} of claims to catch 60% of irregular ones")

In [ ]:
# The same for logistic regression.
share, caught = gain_curve(y_test, p_lr)
first = np.argmax(caught >= 0.60)
print(f"Logistic regression: check {share[first]:.1%} of claims to catch 60% of irregular ones")

**(2)** A common surprise is a negative weight on `months_contributed`: members who have contributed for longer are less
likely to have irregular claims, which makes sense. If you find something that doesn't make sense, check whether that
input moves together with another one. When two inputs carry the same information, their weights can take odd signs
that don't mean much on their own.

**(3)** Without gender. First make a category list without `gender`, then train and measure exactly as before:

In [ ]:
categorical_no_gender = ["benefit_type", "channel", "employer_sector"]

no_gender = Pipeline([
    ("prep", make_preprocessor(NUMERIC, categorical_no_gender, scale_numbers=False)),
    ("model", make_xgboost(spw)),
])
row_ng, p_ng = evaluate_model("XGBoost without gender", no_gender,
                              train[NUMERIC + categorical_no_gender], y_train,
                              test[NUMERIC + categorical_no_gender], y_test)

comparison = pd.DataFrame([row_xgb, row_ng]).set_index("model")
comparison[["roc_auc", "precision_at_cap", "recall_at_cap"]].round(3)

In [ ]:
# Flag rates by gender, with and without gender as an input.
fairness["flagged_without_gender"] = p_ng >= row_ng["threshold"]
fairness.groupby("gender")[["flagged", "flagged_without_gender"]].mean().round(3)

Removing gender barely changes performance here, which means the model never really needed it. That's a good reason to
leave it out. But flag rates by gender can still differ, because benefit type (maternity) and other inputs are linked to
gender. Removing a column doesn't remove its influence. That's why the fairness check in Section 7 looks at outcomes,
not just at the list of inputs.

**(4)** Known when a claim is lodged: everything except `irregular`, `investigation_opened` and `amount_recovered_tzs`.
Worth double-checking with the system owners:

- `documents_missing_at_lodging`: is it recorded at lodging, or updated later as documents arrive?
- `employer_arrears_months_12m`: is it worked out from payments known **on the claim date**, or from a later copy of
  the data?

Timing questions like these are exactly the ones to ask before trusting any input.